# 01회차 실습 노트북 — 핀테크 산업의 이해와 핀테크 데이터

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/01_fintech_industry.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/01_fintech_industry.html](https://adminmrobo.github.io/fintech_data_analysis/01_fintech_industry.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 은행 고객 데이터 한눈에 보기
신용카드 고객 10,127명의 정보입니다. 한 줄이 고객 한 명, 한 열이 정보 하나예요.

In [2]:
b = load('Revolution/Bank_customers_data.csv')
b['y'] = (b.Attrition_Flag == 'Attrited Customer').astype(int)   # 1 = 이탈 고객
print(b.shape, '이탈 비율:', round(b.y.mean(), 3))
b.head()

(10127, 22) 이탈 비율: 0.161


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,y
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0


## 2. 열마다 이탈 비율이 어떻게 다른가
숫자 열은 5개 구간으로 나눠 구간별 이탈 비율을, 글자 열은 값별 이탈 비율을 봅니다.

In [3]:
for c in ['Gender', 'Income_Category', 'Total_Trans_Ct']:
    s = b[c]
    if not pd.api.types.is_numeric_dtype(s):
        print(b.groupby(c).y.mean().round(3), '\n')
    else:
        print(b.groupby(pd.qcut(s, 5)).y.mean().round(3), '\n')

Gender
F    0.174
M    0.146
Name: y, dtype: float64 

Income_Category
$120K +           0.173
$40K - $60K       0.151
$60K - $80K       0.135
$80K - $120K      0.158
Less than $40K    0.172
Unknown           0.168
Name: y, dtype: float64 

Total_Trans_Ct
(9.999, 41.0]    0.327
(41.0, 61.0]     0.348
(61.0, 73.0]     0.066
(73.0, 83.0]     0.033
(83.0, 139.0]    0.011
Name: y, dtype: float64 



## 3. 대결 재현: 행동 1열 vs 인구통계 16열
거래 횟수 한 열만 쓴 로지스틱 회귀와, 나이·성별·학력·소득·결혼(더미 16열)을 쓴 로지스틱 회귀를 AUC로 비교합니다.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score

def lr_eval(X, y):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=RS, stratify=y)
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(max_iter=2000, class_weight='balanced').fit(sc.transform(Xtr), ytr)
    p = m.predict_proba(sc.transform(Xte))[:, 1]
    return round(roc_auc_score(yte, p), 3), round(recall_score(yte, p >= .5), 3)

demo = pd.get_dummies(b[['Customer_Age', 'Gender', 'Education_Level', 'Income_Category', 'Marital_Status']], drop_first=True).astype(float)
print('인구통계', demo.shape[1], '열  (AUC, 재현율):', lr_eval(demo, b.y))
print('거래 횟수 1열     (AUC, 재현율):', lr_eval(b[['Total_Trans_Ct']], b.y))

인구통계 16 열  (AUC, 재현율): (0.52, 0.555)
거래 횟수 1열     (AUC, 재현율): (0.789, 0.786)


**AI 판단 연습** — 누군가 "열이 많을수록 예측이 좋다"고 하면, 위 결과로 어떻게 반박할 수 있을까요? 열의 개수와 열의 정보량은 다릅니다.